# Train one ACT policy for the tolerance-radius experiment (Colab)

One notebook serves both training runs. The two runs differ in `DATASET_NAME` and `RUN_NAME` (first code cell) and in
nothing else. The procedure around this notebook, with every command, is in `training/README.md`.

**What running it does.** It checks the machine, installs LeRobot, signs in from two uploaded files, loads the dataset
from the Hub and checks it (55 episodes, cameras `front` and `wrist`), starts `lerobot-train` **detached** on this
machine, and waits for the first logged training step. It does not wait for training to end: that takes about five
hours, and nothing here needs a connection to stay open.

**Before running**

1. A runtime with an **A100 GPU**.
2. Two files uploaded to `/content/`. A token is never pasted into a cell: what a cell contains is kept in notebook
   outputs and in command histories.
   - `/content/hf_token_file`: the Hub write token of the account that owns `NAMESPACE`;
   - `/content/wandb_key_file`: the Weights & Biases key.

   Each file holds the bare secret on one line, or a `NAME=value` line (`HF_TOKEN=...`, `WANDB_API_KEY=...`). The
   sign-in cell moves both out of `/content/` and prints neither.
3. The parameters: edit the first code cell, or upload a `/content/run_params.json` that overrides it.

**It is safe to run every cell from top to bottom, and to run it all again.** Each check opens a gate when it passes,
and the launch cell starts nothing unless every gate is open: some ways of running a notebook carry on after a cell
has failed. A second pass finds the trainer already running and starts nothing.

In [ ]:
# @title 1. Parameters (the only cell to edit)
GATES = {}  # each check below records here that it passed; the launch cell starts nothing unless all of them did

# Between the two runs, change DATASET_NAME and RUN_NAME, and nothing else.
NAMESPACE = "NAMESPACE"  # the Hub account or organisation that owns the dataset and the policy
DATASET_NAME = "tolrad-clustered-v1"  # the other run: "tolrad-spread-v1"
RUN_NAME = "act-tolrad-clustered-v1"  # the other run: "act-tolrad-spread-v1"
STEPS = 80_000  # protocol: a fixed number of steps, the same for both runs
SAVE_FREQ = 10_000  # protocol: a checkpoint every 10,000 steps, pushed to the Hub

# Rarely changed.
MODE = "fresh"  # "fresh": a new run | "check": every check, start nothing | "resume_hub" | "resume_local"
EXPECTED_EPISODES = 55  # protocol: every recorded episode is trained on
EXPECTED_FRAMES = None  # the frame count the dataset check printed at the rig; None skips this comparison
WANDB_PROJECT = "tolrad-act"
WANDB_ENTITY = None  # None: the default entity of the uploaded key
FOLLOW = False  # True only in a browser tab: the follow cell then runs until the trainer exits
SEED = None  # None: LeRobot's default, 1000, as in the paper's two runs | a number: a further run, under a RUN_NAME of its own

import json
import os
import re

PARAMS_FILE = "/content/run_params.json"
_SETTABLE = (
    "NAMESPACE", "DATASET_NAME", "RUN_NAME", "STEPS", "SAVE_FREQ", "MODE",
    "EXPECTED_EPISODES", "EXPECTED_FRAMES", "WANDB_PROJECT", "WANDB_ENTITY", "FOLLOW", "SEED",
)  # fmt: skip
if os.path.exists(PARAMS_FILE):
    with open(PARAMS_FILE, encoding="utf-8") as _handle:
        _overrides = json.load(_handle)
    _unknown = sorted(set(_overrides) - set(_SETTABLE))
    assert not _unknown, f"{PARAMS_FILE} sets unknown parameter(s): {_unknown}"
    globals().update(_overrides)
    print(f"parameters taken from {PARAMS_FILE}: {sorted(_overrides)}")

_NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*$")
assert NAMESPACE != "NAMESPACE" and _NAME.match(str(NAMESPACE)), (
    "set NAMESPACE to the Hub account or organisation that owns the repositories"
)
assert _NAME.match(str(DATASET_NAME)) and _NAME.match(str(RUN_NAME)), (
    "DATASET_NAME and RUN_NAME are repository names, written without the namespace"
)
assert DATASET_NAME in RUN_NAME, (
    f"RUN_NAME ({RUN_NAME}) does not contain DATASET_NAME ({DATASET_NAME}): were both changed for this run? "
    "A run name left from the other run would push this policy into the other policy's repository."
)
assert MODE in ("fresh", "check", "resume_hub", "resume_local"), f"unknown MODE {MODE!r}"
assert SEED is None or (type(SEED) is int and SEED >= 0), "SEED is a whole number, or None for LeRobot's default"
assert SEED is None or RUN_NAME != f"act-{DATASET_NAME}", (
    f"a run with a seed of its own needs a RUN_NAME of its own (for example act-{DATASET_NAME}-s{SEED}): "
    f"act-{DATASET_NAME} is the paper's run, trained with LeRobot's default seed"
)
assert type(STEPS) is int and type(SAVE_FREQ) is int and 0 < SAVE_FREQ <= STEPS and STEPS % SAVE_FREQ == 0, (
    "STEPS must be a whole number of SAVE_FREQ, so that both runs have checkpoints at the same steps"
)
assert type(EXPECTED_EPISODES) is int and EXPECTED_EPISODES > 0, "EXPECTED_EPISODES is a count of episodes"
assert EXPECTED_FRAMES is None or type(EXPECTED_FRAMES) is int, "EXPECTED_FRAMES is a count of frames, or None"
assert type(FOLLOW) is bool, "FOLLOW is True or False"

print(f"dataset {DATASET_NAME} | run {RUN_NAME} | {STEPS:,} steps, a checkpoint every {SAVE_FREQ:,} | mode {MODE}")
GATES["parameters"] = True

In [ ]:
# @title 2. Settings shared by both runs (not to edit: they are the experiment)
LEROBOT_VERSION = "0.6.1"  # the version the earlier run was trained with; a resumed run must use the same one
EXPECTED_GPU = "A100"  # both runs on the same GPU type
BATCH_SIZE = 32
NUM_WORKERS = 8
LOG_FREQ = 200
EVAL_SPLIT = 0.0  # no episode is held out: all of them are trained on
EVAL_STEPS = 0  # so there is no evaluation pass (LeRobot refuses a positive value without a split)
MAX_EVAL_SAMPLES = 512  # the earlier run's value; it has no effect without a held-out split

WORK_DIR = "/content"
DATASET_REPO = f"{NAMESPACE}/{DATASET_NAME}"
POLICY_REPO = f"{NAMESPACE}/{RUN_NAME}"
OUTPUT_DIR = f"{WORK_DIR}/train/{RUN_NAME}"
TRAIN_LOG = f"{WORK_DIR}/train.log"
PID_FILE = f"{WORK_DIR}/train.pid"
FREEZE_FILE = f"{WORK_DIR}/pip-freeze.txt"
HF_TOKEN_UPLOAD = f"{WORK_DIR}/hf_token_file"
WANDB_KEY_UPLOAD = f"{WORK_DIR}/wandb_key_file"
WANDB_KEY_STORE = os.path.expanduser("~/.config/wandb_api_key")

# Planning figures from the earlier run on this GPU type: its average speed over the whole run, and what an A100
# hour cost in compute units then. They size the estimates printed here and are used for nothing else.
EARLIER_STEPS_PER_S = 4.33
EARLIER_UNITS_PER_HOUR = 5.3
_hours = STEPS / EARLIER_STEPS_PER_S / 3600
print(f"{STEPS:,} steps x batch {BATCH_SIZE} = {STEPS * BATCH_SIZE:,} samples")
print(
    f"estimate at {EARLIER_STEPS_PER_S} steps/s: {_hours:.1f} h, about {_hours * EARLIER_UNITS_PER_HOUR:.0f} compute "
    f"units; a checkpoint about every {SAVE_FREQ / EARLIER_STEPS_PER_S / 60:.0f} min"
)
GATES["settings"] = True

In [ ]:
# @title 3. Helpers used by the cells below (they only define functions)
from pathlib import Path


def read_secret(path, names):
    """The secret in an uploaded file: a bare value on one line, or the value of a `NAME=value` line."""
    lines = [line.strip() for line in Path(path).read_text(encoding="utf-8").splitlines()]
    lines = [line for line in lines if line and not line.startswith("#")]
    for line in lines:
        name, separator, value = line.partition("=")
        if separator and name.strip().removeprefix("export ").strip() in names:
            return value.strip().strip("\"'")
    if len(lines) == 1 and "=" not in lines[0] and " " not in lines[0]:
        return lines[0]
    raise ValueError(f"{path} holds neither one bare value nor a line that sets one of {names}")


def colab_secret(name):
    """A secret from Colab's own store. It answers only in a browser tab; anywhere else this returns None."""
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def checkpoint_steps(files):
    """The step folders under checkpoints/ among a repository's file names, lowest step first."""
    parts = (name.split("/") for name in files)
    steps = {p[1] for p in parts if len(p) >= 3 and p[0] == "checkpoints" and p[1].isdigit()}
    return sorted(steps, key=int)


def saved_steps(output_dir):
    """The step folders under an output directory's checkpoints/, lowest step first."""
    folder = os.path.join(output_dir, "checkpoints")
    names = os.listdir(folder) if os.path.isdir(folder) else []
    return sorted((name for name in names if name.isdigit()), key=int)


def trainer_pids(proc="/proc"):
    """The process ids of the lerobot-train commands running on this machine."""
    found = []
    for entry in os.listdir(proc) if os.path.isdir(proc) else []:
        if entry.isdigit():
            try:
                with open(f"{proc}/{entry}/cmdline", "rb") as handle:
                    if b"lerobot-train" in handle.read():
                        found.append(int(entry))
            except OSError:
                pass
    return sorted(found)


def read_log(path=None):
    """The trainer's log as text: progress-bar redraws become lines, and colour codes are removed."""
    path = path or TRAIN_LOG
    if not os.path.exists(path):
        return ""
    with open(path, encoding="utf-8", errors="replace") as handle:
        text = handle.read()
    return re.sub(r"\x1b\[[0-9;]*m", "", text.replace("\r", "\n"))


def log_tail(lines=30):
    """The last lines of the trainer's log."""
    kept = [line for line in read_log().splitlines() if line.strip()]
    return "\n".join(kept[-lines:]) or "(the log is empty)"


def log_status(log):
    """What the trainer has logged since it was last started, read from the log's text."""
    last_start = log.rfind("===== launch ")
    text = log[last_start:] if last_start >= 0 else log  # an earlier start's lines say nothing about this one
    wandb_run = re.search(r"https://wandb\.ai/([^/\s]+)/([^/\s]+)/runs/([A-Za-z0-9_-]+)", text)
    episodes = re.search(r"dataset\.num_episodes=(\d+)", text)
    frames = re.search(r"dataset\.num_frames=(\d+)", text)
    progress = re.findall(r"(\d+)/(\d+) \[([^\]]*step[^\]]*)\]", text)
    return {
        "launches": re.findall(r"===== launch (\S+) mode=(\S+) run=(\S+) =====", log),
        "wandb_run": "/".join(wandb_run.groups()) if wandb_run else None,
        "episodes": int(episodes.group(1)) if episodes else None,
        "frames": int(frames.group(1)) if frames else None,
        "held_out": re.findall(r"Train/eval split:.*", text),
        "resumed": re.findall(r"Resuming data order.*", text),
        "step_lines": re.findall(r"step:\d\S* smpl:.*", text),
        "progress": progress[-1] if progress else None,
        "checkpoints": [int(step) for step in re.findall(r"Checkpoint policy after step (\d+)", text)],
        "ended": "End of training" in text,
        "pushed": "Model pushed to" in text,
    }


GATES["helpers"] = True

In [ ]:
# @title 4. The machine: Python, GPU, processors, disk
import shutil
import subprocess
import sys

print("python:", sys.version.split()[0])
assert sys.version_info >= (3, 12), (
    f"LeRobot {LEROBOT_VERSION} needs Python 3.12 or newer; this runtime has {sys.version.split()[0]}"
)

_smi = shutil.which("nvidia-smi")
GPU_NAME = ""
if _smi:
    GPU_NAME = subprocess.run(
        [_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
    ).stdout.strip()
_free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print("gpu:", GPU_NAME or "NONE")
print("processors:", os.cpu_count(), "| free disk, GB:", round(_free_gb, 1))

assert GPU_NAME, "no GPU on this runtime: change the runtime type to a GPU and start again"
assert EXPECTED_GPU in GPU_NAME, (
    f"this runtime has '{GPU_NAME}', not an {EXPECTED_GPU}. Both runs must use the same GPU type: "
    "release this runtime and ask for the right one."
)
assert _free_gb > 15, "less than 15 GB free: the checkpoints, the dataset and the packages need about 10 GB"
GATES["machine"] = True

In [ ]:
# @title 5. Install LeRobot
# The [training] extra brings `accelerate`, which lerobot-train needs. The version is pinned: both runs, and any
# resumed run, must use the same one.
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"lerobot[training]=={LEROBOT_VERSION}"],
    capture_output=True,
    text=True,
)
print((_pip.stdout + _pip.stderr)[-4000:].strip() or "pip had nothing to say")
assert _pip.returncode == 0, f"pip could not install lerobot[training]=={LEROBOT_VERSION}: read its messages above"
GATES["install"] = True

In [ ]:
# @title 6. The installed environment, and its fingerprint for the record
import hashlib
import importlib.metadata as md


def installed_version(distribution):
    """The installed version of a package, or None when it is not installed."""
    try:
        return md.version(distribution)
    except md.PackageNotFoundError:
        return None


# A module imported before pip replaced it would run old code against new files: restart the kernel if so.
_stale = []
for _module in ("numpy", "torch", "torchvision", "pandas", "pyarrow", "huggingface_hub", "datasets"):
    _loaded = getattr(sys.modules.get(_module), "__version__", None)
    _installed = installed_version(_module)
    if _loaded and _installed and _loaded.split("+")[0] != _installed.split("+")[0]:
        _stale.append(f"{_module} {_loaded} is loaded, {_installed} is installed")
assert not _stale, "restart the kernel and run again from the top: " + "; ".join(_stale)

import lerobot
import torch

assert lerobot.__version__ == LEROBOT_VERSION, f"lerobot {lerobot.__version__} is installed, not {LEROBOT_VERSION}"
assert not torch.__version__.endswith("+cpu"), (
    f"a CPU-only torch is installed ({torch.__version__}): training would fall back to the processor and take days"
)
assert torch.cuda.is_available(), "torch cannot see the GPU: restart the kernel and run again from the top"

_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
with open(FREEZE_FILE, "w", encoding="utf-8") as _handle:
    _handle.write(_freeze)
_freeze_hash = hashlib.sha256("\n".join(sorted(_freeze.splitlines())).encode()).hexdigest()[:16]
ENV_LINE = (
    f"python={sys.version.split()[0]} lerobot={lerobot.__version__} torch={torch.__version__} "
    f"torchvision={installed_version('torchvision')} accelerate={installed_version('accelerate')} "
    f"numpy={installed_version('numpy')} cuda={torch.version.cuda} gpu='{torch.cuda.get_device_name(0)}' "
    f"processors={os.cpu_count()} pip_freeze_sha256={_freeze_hash}"
)
print("ENV", ENV_LINE)
print(f"Both runs must print the same ENV line. If only the hash differs, compare the two {FREEZE_FILE} files.")
GATES["environment"] = True

In [ ]:
# @title 7. Sign in from the uploaded files (nothing secret is printed)
from huggingface_hub import HfApi, get_token, login, whoami

# The Hub token: the uploaded file, else a token already stored on this machine, else Colab's secret store.
if os.path.exists(HF_TOKEN_UPLOAD):
    login(token=read_secret(HF_TOKEN_UPLOAD, ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")))
    os.remove(HF_TOKEN_UPLOAD)
elif not get_token():
    _secret = colab_secret("HF_TOKEN")
    assert _secret, f"no Hub token: upload it to {HF_TOKEN_UPLOAD} (training/README.md, step 3)"
    login(token=_secret)

# The Weights & Biases key: kept in a file only this user can read, and handed to the trainer by the launch cell.
_secret = None
if os.path.exists(WANDB_KEY_UPLOAD):
    _secret = read_secret(WANDB_KEY_UPLOAD, ("WANDB_API_KEY",))
    os.remove(WANDB_KEY_UPLOAD)
elif not os.path.exists(WANDB_KEY_STORE):
    _secret = colab_secret("WANDB_API_KEY")
    assert _secret, f"no Weights & Biases key: upload it to {WANDB_KEY_UPLOAD} (training/README.md, step 3)"
if _secret:
    os.makedirs(os.path.dirname(WANDB_KEY_STORE), exist_ok=True)
    _descriptor = os.open(WANDB_KEY_STORE, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with open(_descriptor, "w", encoding="utf-8") as _handle:
        _handle.write(_secret.strip())
_secret = None
assert len(Path(WANDB_KEY_STORE).read_text(encoding="utf-8").strip()) >= 20, "the stored W&B key is too short to be one"

# The token must be able to write where the checkpoints go. A wrong account would otherwise stop the run at its
# first checkpoint, 10,000 steps in.
api = HfApi()
_me = whoami()
_owners = {_me["name"], *(org["name"] for org in _me.get("orgs", []))}
assert NAMESPACE in _owners, "the Hub token belongs neither to NAMESPACE nor to a member of that organisation"
_role = _me.get("auth", {}).get("accessToken", {}).get("role")
assert _role != "read", "the Hub token is read-only: the run could not push its checkpoints"
print("the Hub token can act for the namespace | token role:", _role)

assert api.dataset_info(DATASET_REPO).private, "the dataset repository is public: make it private before going on"
if MODE == "check":
    print("mode check: the policy repository is neither created nor read")
else:
    if MODE == "fresh":
        # The call the trainer itself makes at its first checkpoint: made now, it proves the token can write here.
        api.create_repo(POLICY_REPO, repo_type="model", private=True, exist_ok=True)
    assert api.model_info(POLICY_REPO).private, "the policy repository is public: make it private before going on"
    _on_hub = checkpoint_steps(api.list_repo_files(POLICY_REPO, repo_type="model"))
    print("checkpoints on the Hub:", _on_hub or "none")
    if MODE == "fresh" and not trainer_pids():
        assert not _on_hub, (
            f"the policy repository already holds checkpoints {_on_hub}. A new run must start in an empty "
            "repository. To continue that run set MODE to 'resume_hub' (on a new machine) or 'resume_local' (on the "
            "machine that holds it); for a different run use another RUN_NAME."
        )
    if MODE == "resume_hub" and not trainer_pids():
        assert _on_hub, "nothing to resume: the policy repository holds no checkpoint"
GATES["sign_in"] = True

In [ ]:
# @title 8. The dataset, loaded from the Hub on this clean machine
from lerobot.datasets import LeRobotDataset

try:
    ds = LeRobotDataset(DATASET_REPO)
except Exception as _error:
    if "codebase version" in str(_error):
        print("The dataset repository has no version tag: create the tag (training/README.md, step 2) and run again.")
    raise

_cameras = sorted(ds.meta.camera_keys)
_tasks = list(ds.meta.tasks.index) if hasattr(ds.meta.tasks, "index") else list(ds.meta.tasks)
print(f"{ds.num_episodes} episodes, {ds.num_frames} frames, {ds.fps} fps")
print("cameras:", _cameras)
print("task:", _tasks)

_first, _last = ds[0], ds[ds.num_frames - 1]  # decoded here, at both ends of the dataset
for _key in _cameras:
    print(f"  {_key}: {tuple(_first[_key].shape)}")
print("  action:", tuple(_first["action"].shape), "| state:", tuple(_first["observation.state"].shape))

assert ds.num_episodes == EXPECTED_EPISODES, (
    f"{ds.num_episodes} episodes on the Hub, {EXPECTED_EPISODES} expected: this is not the recorded dataset"
)
assert EXPECTED_FRAMES is None or ds.num_frames == EXPECTED_FRAMES, (
    f"{ds.num_frames} frames on the Hub, {EXPECTED_FRAMES} counted at the rig"
)
assert _cameras == ["observation.images.front", "observation.images.wrist"], f"unexpected cameras: {_cameras}"
assert ds.fps == 30, f"{ds.fps} frames a second, 30 expected"
for _key in _cameras:
    assert tuple(_first[_key].shape) == (3, 480, 640), f"{_key}: the first frame is not 640x480"
    assert tuple(_last[_key].shape) == (3, 480, 640), f"{_key}: the last frame is not 640x480"
assert tuple(_first["action"].shape) == (6,), "the action is not six joints"
assert tuple(_first["observation.state"].shape) == (6,), "the state is not six joints"
assert len(_tasks) == 1, f"{len(_tasks)} task strings in the dataset, 1 expected: {_tasks}"
DATASET_EPISODES, DATASET_FRAMES = ds.num_episodes, ds.num_frames
GATES["dataset"] = True

In [ ]:
# @title 9. Can the processors decode video fast enough? (informational: it opens no gate)
# Frames are video, decoded on the processors. Random access is what a shuffled data loader does.
import random
import time

random.seed(0)
_indices = [random.randrange(ds.num_frames) for _ in range(40)]
_ = ds[_indices[0]]  # the first touch is not timed
_start = time.time()
for _index in _indices:
    _ = ds[_index]
_per_sample = (time.time() - _start) / len(_indices)
_ceiling = NUM_WORKERS / (_per_sample * BATCH_SIZE)
print(
    f"{_per_sample * 1000:.1f} ms a sample for one worker: about {_ceiling:.0f} steps/s "
    f"from {NUM_WORKERS} workers at batch {BATCH_SIZE}"
)
if _ceiling > 2 * EARLIER_STEPS_PER_S:
    print(f"The GPU ran {EARLIER_STEPS_PER_S} steps/s in the earlier run: the data loader will keep up.")
else:
    print(f"SLOW: the GPU ran {EARLIER_STEPS_PER_S} steps/s in the earlier run and the data loader may hold it back.")
    print("The run would still be correct, only slower. Write this line into the record.")

## What is about to run

`lerobot-train`, with the flags printed by the next cell. Against the earlier run, four values change, all of them set
by the protocol:

| Flag | Earlier run | Now | Why |
|---|---|---|---|
| `--dataset.eval_split` | 0.1 | **0.0** | every episode is trained on; none is held out |
| `--eval_steps` | 5000 | **0** | with no held-out episodes there is no evaluation loss; LeRobot refuses a positive value without a split |
| `--steps` | 100000 | **80000** | a fixed number of steps, decided before any result |
| `--save_freq` | 5000 | **10000** | checkpoints at 10,000, 20,000, ... and 80,000, the step that is evaluated |

`--max_eval_samples=512` is kept from the earlier run and does nothing without a held-out split. ACT has no
learning-rate schedule, so stopping at 80,000 changes nothing about the steps before it.

Each checkpoint is pushed to the policy repository as `checkpoints/<step>/` when it is written. If this machine is
lost, the run continues from the last one: `MODE = "resume_hub"` on a new machine.

In [ ]:
# @title 10. The training command (printed here; the next cell starts it)
import shlex


def build_command(mode):
    """The trainer's command line: for a new run, or to resume one from this machine or from the Hub."""
    trainer = shutil.which("lerobot-train") or "lerobot-train"
    if mode == "resume_local":
        last = f"{OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json"
        return [trainer, f"--config_path={last}", "--resume=true"]
    if mode == "resume_hub":
        return [trainer, f"--config_path={POLICY_REPO}", "--resume=true", f"--output_dir={OUTPUT_DIR}"]
    command = [
        trainer,
        f"--dataset.repo_id={DATASET_REPO}",
        f"--dataset.eval_split={EVAL_SPLIT}",
        "--policy.type=act",
        "--policy.device=cuda",
        f"--policy.repo_id={POLICY_REPO}",
        "--policy.private=true",
        f"--output_dir={OUTPUT_DIR}",
        f"--job_name={RUN_NAME}",
        f"--steps={STEPS}",
        f"--batch_size={BATCH_SIZE}",
        f"--num_workers={NUM_WORKERS}",
        f"--save_freq={SAVE_FREQ}",
        "--save_checkpoint_to_hub=true",
        f"--eval_steps={EVAL_STEPS}",
        f"--max_eval_samples={MAX_EVAL_SAMPLES}",
        f"--log_freq={LOG_FREQ}",
        "--wandb.enable=true",
        f"--wandb.project={WANDB_PROJECT}",
    ]
    if WANDB_ENTITY:
        command.append(f"--wandb.entity={WANDB_ENTITY}")
    if SEED is not None:
        command.append(f"--seed={SEED}")
    return command


assert EVAL_SPLIT == 0.0 and EVAL_STEPS == 0, "the protocol trains on every episode: no split, no evaluation pass"
CMD = build_command(MODE)
print(" \\\n    ".join(shlex.quote(part) for part in CMD))
GATES["command"] = True

In [ ]:
# @title 11. Start the trainer, detached from this notebook
import datetime
import time

REQUIRED_GATES = (
    "parameters", "settings", "helpers", "machine", "install", "environment", "sign_in", "dataset", "command",
)  # fmt: skip
_closed = [gate for gate in REQUIRED_GATES if not GATES.get(gate)]

if trainer_pids():
    print(f"A trainer is already running (process {trainer_pids()}): nothing was started.")
    print("The next cell reports on it.")
elif _closed:
    raise RuntimeError(
        f"NOT STARTED. These checks have not passed: {_closed}. Fix the first failing cell above and run again."
    )
elif MODE == "check":
    print("MODE is 'check': every check passed, and nothing was started.")
else:
    _saved = saved_steps(OUTPUT_DIR)
    if MODE == "resume_local":
        if not _saved:
            raise RuntimeError(f"NOT STARTED. {OUTPUT_DIR} holds no checkpoint to resume from: use MODE 'resume_hub'.")
    elif _saved:
        raise RuntimeError(
            f"NOT STARTED. {OUTPUT_DIR} already holds checkpoints {_saved}: this machine has the run. "
            "Set MODE to 'resume_local' to continue it."
        )
    elif os.path.isdir(OUTPUT_DIR):
        print(f"removing {OUTPUT_DIR}: an earlier start left it behind, with no checkpoint in it")
        shutil.rmtree(OUTPUT_DIR)

    _environment = dict(os.environ)
    _environment["WANDB_API_KEY"] = Path(WANDB_KEY_STORE).read_text(encoding="utf-8").strip()
    _now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    with open(TRAIN_LOG, "ab") as _log:
        _log.write(f"\n===== launch {_now} mode={MODE} run={RUN_NAME} =====\n".encode())
        _log.flush()
        TRAINER = subprocess.Popen(
            CMD,
            stdout=_log,
            stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL,
            cwd=WORK_DIR,
            env=_environment,
            start_new_session=True,  # its own session: it outlives this cell, this kernel and any connection
        )
    _environment = None
    with open(PID_FILE, "w", encoding="utf-8") as _handle:
        _handle.write(str(TRAINER.pid))

    time.sleep(60)  # a wrong key, a missing repository or a bad flag ends the trainer within seconds
    if TRAINER.poll() not in (None, 0):
        print(log_tail())
        raise RuntimeError(
            f"NOT RUNNING. The trainer stopped at once with exit status {TRAINER.returncode}: read the log tail above."
        )
    print(f"STARTED at {_now} (this machine's clock), process {TRAINER.pid}, log {TRAIN_LOG}")
    if TRAINER.poll() == 0:
        print("It has already ended without an error: the next cell says what it did.")

In [ ]:
# @title 12. Wait for the first logged step, then report (run again at any time for the current state)
if MODE == "check":
    print("mode check: there is no trainer to wait for")
else:
    _deadline = time.time() + 15 * 60
    while True:
        STATUS = log_status(read_log())
        _alive = trainer_pids()
        if STATUS["step_lines"] or STATUS["ended"] or not _alive or time.time() > _deadline:
            break
        time.sleep(15)

    print("trainer running:", bool(_alive), _alive or "")
    if STATUS["step_lines"]:
        print("first logged step: ", STATUS["step_lines"][0])
        print("latest logged step:", STATUS["step_lines"][-1])
    if STATUS["progress"]:
        print("progress since the last start: {}/{} [{}]".format(*STATUS["progress"]))
    print("checkpoints announced since the last start:", STATUS["checkpoints"] or "none yet")
    print("the trainer counted:", STATUS["episodes"], "episodes,", STATUS["frames"], "frames")
    if STATUS["resumed"]:
        print("resumed:", STATUS["resumed"][-1])
    if STATUS["ended"]:
        print("the log says 'End of training'" + (", and the final policy was pushed" if STATUS["pushed"] else ""))
    if STATUS["wandb_run"]:
        print(f"for the watchers:  WANDB_RUN={STATUS['wandb_run']}")

    if STATUS["held_out"]:
        raise RuntimeError(
            f"STOP THIS RUN. The trainer held episodes out, against the protocol: {STATUS['held_out'][0]}"
        )
    if STATUS["episodes"] is not None and STATUS["episodes"] != EXPECTED_EPISODES:
        raise RuntimeError(
            f"STOP THIS RUN. The trainer counted {STATUS['episodes']} episodes, not {EXPECTED_EPISODES}."
        )
    if not STATUS["step_lines"] and not STATUS["ended"]:
        print(log_tail())
        _why = "it is still starting, slowly" if _alive else "the trainer has stopped"
        raise RuntimeError(f"NO LOGGED STEP: {_why}. Read the log tail above before trusting this run.")
    if not _alive and not STATUS["ended"]:
        print(log_tail())
        raise RuntimeError("THE TRAINER HAS STOPPED without 'End of training': read the log tail above.")
    if STATUS["episodes"] is None:
        print("The log does not state the episode count in the expected form: check by eye that none was held out.")

    _started = STATUS["launches"][-1][0] if STATUS["launches"] else "unknown"
    print()
    print("RUN RECORD (it names no account: it can go into the journal as it is)")
    print(f"  dataset {DATASET_NAME}: {DATASET_EPISODES} episodes, {DATASET_FRAMES} frames, loaded from the Hub here")
    print(f"  run {RUN_NAME}: {STEPS:,} steps, a checkpoint every {SAVE_FREQ:,}, mode {MODE}")
    print(f"  last start {_started} (this machine's clock)")
    print(f"  {ENV_LINE}")
    print("  first logged step:", STATUS["step_lines"][0] if STATUS["step_lines"] else "none")

In [ ]:
# @title 13. Follow the run until it ends (only when FOLLOW is True: it keeps a browser session busy)
if FOLLOW and MODE != "check":
    while trainer_pids():
        time.sleep(300)
        _lines = log_status(read_log())["step_lines"]
        print(time.strftime("%H:%M:%S"), _lines[-1] if _lines else "(no logged step yet)")
    STATUS = log_status(read_log())
    if STATUS["ended"]:
        print("The trainer has exited, and the log says 'End of training'.")
    else:
        print(log_tail())
        print("THE TRAINER HAS EXITED WITHOUT 'End of training': read the log tail above.")
else:
    print("FOLLOW is off (or MODE is 'check'): this cell did nothing.")

In [ ]:
# @title 14. What the Hub holds for this run (run again at any time)
if MODE == "check":
    print("mode check: there is no policy repository to look at")
else:
    _files = api.list_repo_files(POLICY_REPO, repo_type="model")
    _steps = checkpoint_steps(_files)
    _expected = [f"{step:06d}" for step in range(SAVE_FREQ, STEPS + 1, SAVE_FREQ)]
    _whole = [
        step
        for step in _steps
        if f"checkpoints/{step}/pretrained_model/model.safetensors" in _files
        and f"checkpoints/{step}/pretrained_model/train_config.json" in _files
        and f"checkpoints/{step}/training_state/training_step.json" in _files
    ]
    print("checkpoints on the Hub:", _steps or "none yet")
    if _whole != _steps:
        print("INCOMPLETE on the Hub (a file is missing):", [step for step in _steps if step not in _whole])
    print("still to come:", [step for step in _expected if step not in _steps] or "none")
    print("final policy at the top of the repository:", "model.safetensors" in _files)
    if not _steps:
        print(f"The first checkpoint is due about {SAVE_FREQ / EARLIER_STEPS_PER_S / 60:.0f} min after the first step.")

## After the start

- **Leave this machine alone.** The trainer needs no connection. Do not poll it from the command line: an expired
  connection there can end the session that keeps this machine alive (`training/README.md`, step 4).
- **Watch from outside**: the Weights & Biases run (the step and the training loss) and the policy repository on the
  Hub (a new `checkpoints/<step>/` about every 38 minutes). A watcher counts once it has shown this run's own numbers.
- **When step 80,000 is on the Hub**, release this machine: an idle A100 still costs compute units.
- **If this machine is lost**, start a new one and run this notebook with `MODE = "resume_hub"` and the same names:
  the run continues from its last checkpoint on the Hub. Write the interruption into the journal.